In [2]:
import os
import re
import time
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv


load_dotenv()

True

In [3]:
os.environ["HF_HOME"] = "/data/oussama/hf_cache"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/data/oussama/hf_cache/hub"
os.environ["TRANSFORMERS_CACHE"] = "/data/oussama/hf_cache/transformers"

In [5]:
# check if the GPUs are available
import sys, subprocess, torch

print("Python:", sys.executable)
print("torch:", torch.__version__)
print("torch file:", torch.__file__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))

print(subprocess.check_output(["nvidia-smi"], text=True))

Python: /data/oussama/NYU/ve_qwen/bin/python
torch: 2.7.1+cu118
torch file: /data/oussama/NYU/ve_qwen/lib/python3.12/site-packages/torch/__init__.py
CUDA build: 11.8
CUDA available: True
GPU count: 4
0 Tesla P100-PCIE-16GB
1 Tesla P100-PCIE-16GB
2 Tesla P100-PCIE-16GB
3 Tesla P100-PCIE-16GB
Sat Jun 20 21:45:29 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.309.01             Driver Version: 535.309.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla P100-PCIE-16GB           Off | 00

# Qwen3.6

In [6]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

def run_qwen36_35b_a3b_local_on_questions(
    input_csv,
    output_csv,
    prompt_col="Prompt",
    model_id="Qwen/Qwen3.6-35B-A3B",
    question_id_col=None,
    gold_col=None,
    level_col="level",
    medical_specialty_col="medical_specialty",
    max_retries=5,
    save_every=1,
    max_new_tokens=32,
    cache_dir="/data/oussama/hf_cache",
    device_map="auto",
):
    """
    Run Qwen3.6-35B-A3B locally using Transformers and save predictions.

    Output CSV columns:
    - question_id
    - gold
    - prediction
    - raw_output
    - correct
    - level
    - medical_specialty
    - model
    - stop_reason
    - refusal_category
    - refusal_type
    """

    input_csv = Path(input_csv)
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    print(f"Loading local Qwen model: {model_id}")

    processor = AutoProcessor.from_pretrained(
        model_id,
        cache_dir=cache_dir,
        trust_remote_code=True,
    )

    # Newer Transformers supports dtype="auto"; older versions may require torch_dtype="auto".
    model_kwargs = dict(
        cache_dir=cache_dir,
        device_map=device_map,
        trust_remote_code=True,
    )
    try:
        qwen_model = AutoModelForMultimodalLM.from_pretrained(
            model_id,
            dtype="auto",
            **model_kwargs,
        )
    except TypeError:
        qwen_model = AutoModelForMultimodalLM.from_pretrained(
            model_id,
            torch_dtype="auto",
            **model_kwargs,
        )

    qwen_model.eval()

    df = pd.read_csv(input_csv, encoding="utf-8-sig")

    if prompt_col not in df.columns:
        raise ValueError(f"Missing prompt column: {prompt_col}")

    if gold_col is None:
        raise ValueError(
            "Could not find the gold answer column. "
            "Please pass gold_col='your_column_name'."
        )

    output_columns = [
        "question_id",
        "gold",
        "prediction",
        "raw_output",
        "correct",
        "level",
        "medical_specialty",
        "model",
        "stop_reason",
        "refusal_category",
        "refusal_type",
    ]

    def clean_value(value):
        if pd.isna(value):
            return ""
        return str(value).strip()

    def normalize_gold(value):
        text = clean_value(value).upper()
        match = re.search(r"\b([A-F])\b", text)
        return match.group(1) if match else text[:1]

    def extract_prediction(raw_output, allowed_letters=None):
        text = clean_value(raw_output).upper()

        if allowed_letters:
            allowed_letters = [x.upper() for x in allowed_letters]
        else:
            allowed_letters = ["A", "B", "C", "D", "E", "F"]

        if text in allowed_letters:
            return text

        # Handles outputs like {"answer": "C"} or answer: C.
        answer_pattern = r"[\"']?ANSWER[\"']?\s*[:=]\s*[\"']?(" + "|".join(map(re.escape, allowed_letters)) + r")[\"']?"
        match = re.search(answer_pattern, text)
        if match:
            return match.group(1)

        pattern = r"\b(" + "|".join(map(re.escape, allowed_letters)) + r")\b"
        match = re.search(pattern, text)
        if match:
            return match.group(1)

        if text and text[0] in allowed_letters:
            return text[0]

        return ""

    def parse_qwen_output(decoded_text):
        """Clean Qwen output and remove thinking/special-token artifacts."""
        text = clean_value(decoded_text)

        # Remove Qwen thinking blocks if they appear.
        text = re.sub(r"<think>.*?</think>", " ", text, flags=re.IGNORECASE | re.DOTALL)
        text = re.sub(r"<think>.*", " ", text, flags=re.IGNORECASE | re.DOTALL)

        # Remove common special tokens / chat markers.
        for token in [
            "<|im_start|>",
            "<|im_end|>",
            "<|endoftext|>",
            "<end_of_turn>",
            "<start_of_turn>",
            "assistant",
        ]:
            text = text.replace(token, " ")

        text = clean_value(text)

        # If the model returns JSON, keep it as raw_output but normalize spacing.
        try:
            obj = json.loads(text)
            if isinstance(obj, dict):
                return json.dumps(obj, ensure_ascii=False)
        except Exception:
            pass

        return text

    def model_input_device(model):
        try:
            return next(model.parameters()).device
        except StopIteration:
            return torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def build_inputs(messages):
        base_kwargs = dict(
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            add_generation_prompt=True,
        )

        # Qwen3.6 can be run in non-thinking mode via chat template kwargs.
        # Depending on Transformers version, passing enable_thinking directly may or may not be accepted.
        try:
            inputs = processor.apply_chat_template(
                messages,
                **base_kwargs,
                enable_thinking=False,
            )
        except TypeError:
            inputs = processor.apply_chat_template(
                messages,
                **base_kwargs,
            )

        return inputs.to(model_input_device(qwen_model))

    def call_model(prompt, allowed_letters=None, max_retries=5):
        """
        Calls Qwen3.6 locally.

        Returns:
        {
            "raw_output": str,
            "stop_reason": str,
            "refusal_category": None,
            "refusal_type": None,
        }
        """

        if allowed_letters is None:
            allowed_letters = ["A", "B", "C", "D", "E", "F"]

        allowed_text = ", ".join(allowed_letters)

        system_prompt = (
            "You are evaluating multiple-choice questions from a medical benchmark. "
            "Select the single best answer option. Return only one uppercase letter. "
            "Do not explain."
        )

        user_prompt = (
            f"{prompt}\n\n"
            f"Allowed options: {allowed_text}\n"
            "Return exactly one uppercase letter from the allowed options. "
            "No explanation."
        )

        # Qwen3.6 is a multimodal/chat model; even for text-only prompts, use content blocks.
        messages = [
            {
                "role": "system",
                "content": [
                    {"type": "text", "text": system_prompt},
                ],
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": user_prompt},
                ],
            },
        ]

        last_error = None

        for attempt in range(1, max_retries + 1):
            try:
                inputs = build_inputs(messages)
                input_len = inputs["input_ids"].shape[-1]

                tokenizer = getattr(processor, "tokenizer", None)
                eos_token_id = getattr(tokenizer, "eos_token_id", None)
                pad_token_id = getattr(tokenizer, "pad_token_id", None) or eos_token_id

                with torch.inference_mode():
                    outputs = qwen_model.generate(
                        **inputs,
                        max_new_tokens=max_new_tokens,
                        do_sample=False,
                        pad_token_id=pad_token_id,
                        eos_token_id=eos_token_id,
                    )

                generated_tokens = outputs[0][input_len:]

                decoded = processor.decode(
                    generated_tokens,
                    skip_special_tokens=False,
                )

                raw_output = parse_qwen_output(decoded)

                if raw_output:
                    return {
                        "raw_output": raw_output,
                        "stop_reason": "local_generate",
                        "refusal_category": None,
                        "refusal_type": None,
                    }

                print("Empty local Qwen response:")
                print("decoded:", repr(decoded))
                raise RuntimeError("Qwen returned empty text output.")

            except Exception as e:
                last_error = e

                if attempt == max_retries:
                    break

                wait_time = min(2 ** attempt, 60)
                print(
                    f"Local Qwen error attempt {attempt}/{max_retries}: {repr(e)}. "
                    f"Retrying in {wait_time}s..."
                )
                time.sleep(wait_time)

        raise RuntimeError(
            f"Failed after {max_retries} retries. Last error: {repr(last_error)}"
        )

    # Read already processed question IDs if results CSV exists.
    processed_ids = set()

    if output_csv.exists():
        old_results = pd.read_csv(output_csv, encoding="utf-8-sig")

        if "question_id" in old_results.columns:
            processed_ids = set(old_results["question_id"].astype(str))
        elif "Question_id" in old_results.columns:
            processed_ids = set(old_results["Question_id"].astype(str))

        for col in output_columns:
            if col not in old_results.columns:
                old_results[col] = ""

        old_results = old_results[output_columns]
        results = old_results.to_dict("records")
    else:
        old_results = pd.DataFrame(columns=output_columns)
        results = []

    print(f"Already processed: {len(processed_ids)} questions")

    newly_processed = 0

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        question_id = (
            clean_value(row[question_id_col])
            if question_id_col is not None
            else str(idx)
        )

        if str(question_id) in processed_ids:
            continue

        gold = normalize_gold(row[gold_col])

        group = clean_value(row["Group"]).upper() if "Group" in df.columns else "ABCDEF"
        allowed_letters = [
            letter for letter in group
            if letter in ["A", "B", "C", "D", "E", "F"]
        ]

        if not allowed_letters:
            allowed_letters = ["A", "B", "C", "D", "E", "F"]

        prompt = clean_value(row[prompt_col])

        call_result = call_model(
            prompt=prompt,
            allowed_letters=allowed_letters,
            max_retries=max_retries,
        )

        raw_output = call_result["raw_output"]
        stop_reason = call_result["stop_reason"]
        refusal_category = call_result["refusal_category"]
        refusal_type = call_result["refusal_type"]

        prediction = extract_prediction(raw_output, allowed_letters)
        correct = int(prediction == gold)

        result = {
            "question_id": question_id,
            "gold": gold,
            "prediction": prediction,
            "raw_output": raw_output,
            "correct": correct,
            "level": clean_value(row[level_col]) if level_col in df.columns else "",
            "medical_specialty": (
                clean_value(row[medical_specialty_col])
                if medical_specialty_col in df.columns
                else ""
            ),
            "model": model_id,
            "stop_reason": stop_reason,
            "refusal_category": refusal_category,
            "refusal_type": refusal_type,
        }

        results.append(result)
        processed_ids.add(str(question_id))
        newly_processed += 1

        if save_every and newly_processed % save_every == 0:
            pd.DataFrame(results, columns=output_columns).to_csv(
                output_csv,
                index=False,
                encoding="utf-8-sig",
            )

    results_df = pd.DataFrame(results, columns=output_columns)

    results_df.to_csv(
        output_csv,
        index=False,
        encoding="utf-8-sig",
    )

    return results_df

/data/oussama/NYU/ve_qwen/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# results = run_qwen36_35b_a3b_local_on_questions(
#     input_csv="data/cleaned/medarabench_test_mini_with_prompts.csv",
#     output_csv="results/qwen36_35b_a3b_results_mini.csv",
#     prompt_col="Prompt",
#     model_id="Qwen/Qwen3.6-35B-A3B",
#     question_id_col="Question Number",
#     gold_col="Correct Answer",
#     level_col="Level",
#     medical_specialty_col="Medical Specialty",
#     save_every=1,
# )

Loading local Qwen model: Qwen/Qwen3.6-35B-A3B


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 1026/1026 [39:40<00:00,  2.32s/it] 
Some parameters are on the meta device because they were offloaded to the cpu.


Already processed: 0 questions


100%|██████████| 25/25 [32:07<00:00, 77.10s/it]  


In [ ]:
from run_qwen_local_cpu import run_qwen_local_on_questions

results_df = run_qwen_local_on_questions(
    input_csv="data/cleaned/medarabench_test_with_prompts.csv",
    output_csv="results/qwen36_35b_a3b_results.csv",
    prompt_col="Prompt",
    model_id="Qwen/Qwen3.6-35B-A3B",
    question_id_col="Question Number",
    gold_col="Correct Answer",
    level_col="Level",
    medical_specialty_col="Medical Specialty",
    max_new_tokens=10,
    save_every=1,
)

CPU only: cpu
PyTorch CPU threads: 56 / requested: 56
Loading local Qwen model: Qwen/Qwen3.6-35B-A3B


Loading weights: 100%|██████████| 693/693 [00:00<00:00, 4969.95it/s]


Already processed: 0 questions


 12%|█▏        | 3/25 [04:05<30:00, 81.86s/it]


KeyboardInterrupt: 